# Random Forest

In [16]:
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from rasterio.enums import Resampling
from rasterio import features
import numpy as np
import glob
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression

## Prepare data

In [18]:
# import study area
study_area = rxr.open_rasterio('input-data/Other/study_area_kalimantan.tif').squeeze()

In [ ]:
# import predictor layers (squeeze removed band dimension)

drainage_dist = rxr.open_rasterio("input-data/predictor-layers/drainage-dist.tif").squeeze()
forest_dist = rxr.open_rasterio("input-data/predictor-layers/forest-dist.tif").squeeze()
pre_fire_evi_2m = rxr.open_rasterio("input-data/predictor-layers/evi_2month_pre.tif").squeeze()
pre_fire_evi_2y = rxr.open_rasterio("input-data/predictor-layers/evi_2year_pre.tif").squeeze()
burn_severity = rxr.open_rasterio("input-data/predictor-layers/burn_severity.tif").squeeze()

In [ ]:
# import vegetation metrics
# I guess i renamed them here as outcome layers, in my folder, i hope is not so confusing
evi_recovery_2y = rxr.open_rasterio("output-data/output_EVI_dif/dif_evi_short_2017_minus_2015.tif").squeeze()
evi_recovery_5y = rxr.open_rasterio("output-data/output_EVI_dif/dif_evi_long_2020_minus_2015.tif").squeeze()

In [6]:
# combine in one dataset
ds = xr.Dataset({
    "study_area": study_area,
    "pre_fire_evi_2m": pre_fire_evi_2m,
    "pre_fire_evi_2y": pre_fire_evi_2y,
    "drainage_dist": drainage_dist,
    "forest_dist": forest_dist,
    "burn_severity": burn_severity,
})

In [7]:
# transform to pandas dataframe (only study area pixels)
df = ds.to_dataframe().reset_index()

# remove pixels outside study area
df = df[df.study_area == 1]

In [8]:
# clean data (remove impossible values and missing data)

# Define predictor columns
predictor_cols = ["pre_fire_evi_2m", "pre_fire_evi_2y", "drainage_dist", "forest_dist", "burn_severity"]

# Add outcome layers to dataframe
df_2y = df.copy()
df_5y = df.copy()

# Merge with outcome layers (align by x, y coordinates)
evi_2y_df = evi_recovery_2y.to_dataframe(name="evi_recovery_2y").reset_index()
evi_5y_df = evi_recovery_5y.to_dataframe(name="evi_recovery_5y").reset_index()

df_2y = df_2y.merge(evi_2y_df[["x", "y", "evi_recovery_2y"]], on=["x", "y"], how="inner")
df_5y = df_5y.merge(evi_5y_df[["x", "y", "evi_recovery_5y"]], on=["x", "y"], how="inner")

# Remove rows with any missing values
df_2y_clean = df_2y.dropna(subset=predictor_cols + ["evi_recovery_2y"])
df_5y_clean = df_5y.dropna(subset=predictor_cols + ["evi_recovery_5y"])

# Remove infinite values
df_2y_clean = df_2y_clean[np.isfinite(df_2y_clean[predictor_cols + ["evi_recovery_2y"]]).all(axis=1)]
df_5y_clean = df_5y_clean[np.isfinite(df_5y_clean[predictor_cols + ["evi_recovery_5y"]]).all(axis=1)]

print(f"2-year model: {len(df_2y_clean)} valid pixels")
print(f"5-year model: {len(df_5y_clean)} valid pixels")

2-year model: 7774 valid pixels
5-year model: 7862 valid pixels


In [ ]:
# clean data (remove impossible values 

# remove missing data

In [9]:
# prepare 10-fold CV
from sklearn.model_selection import KFold

# Set up 10-fold cross-validation
kf = KFold(n_splits=10, shuffle=True, random_state=123)

# Prepare X and y for both models
X_2y = df_2y_clean[predictor_cols]
y_2y = df_2y_clean["evi_recovery_2y"]

X_5y = df_5y_clean[predictor_cols]
y_5y = df_5y_clean["evi_recovery_5y"]

print(f"2-year recovery: {X_2y.shape[0]} samples, {X_2y.shape[1]} features")
print(f"5-year recovery: {X_5y.shape[0]} samples, {X_5y.shape[1]} features")

2-year recovery: 7774 samples, 5 features
5-year recovery: 7862 samples, 5 features


## Fit model

In [2]:
# set up model
rf = RandomForestRegressor(
    n_estimators=500, # number of trees
    random_state=123 # for reproducibility
    )

In [ ]:
# 

In [10]:
## Fit model
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# set up models
rf_2y = RandomForestRegressor(
    n_estimators=500,
    random_state=123,
    n_jobs=-1  # use all CPU cores
)

rf_5y = RandomForestRegressor(
    n_estimators=500,
    random_state=123,
    n_jobs=-1
)

# Store CV results
cv_results_2y = {"r2": [], "rmse": [], "mae": []}
cv_results_5y = {"r2": [], "rmse": [], "mae": []}

# 10-fold CV for 2-year model
print("\n=== 2-Year Recovery Model ===")
for fold, (train_idx, test_idx) in enumerate(kf.split(X_2y), 1):
    X_train, X_test = X_2y.iloc[train_idx], X_2y.iloc[test_idx]
    y_train, y_test = y_2y.iloc[train_idx], y_2y.iloc[test_idx]
    
    rf_2y.fit(X_train, y_train)
    y_pred = rf_2y.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    cv_results_2y["r2"].append(r2)
    cv_results_2y["rmse"].append(rmse)
    cv_results_2y["mae"].append(mae)
    
    print(f"Fold {fold}: R² = {r2:.3f}, RMSE = {rmse:.4f}, MAE = {mae:.4f}")

print(f"\nMean R² = {np.mean(cv_results_2y['r2']):.3f} ± {np.std(cv_results_2y['r2']):.3f}")
print(f"Mean RMSE = {np.mean(cv_results_2y['rmse']):.4f} ± {np.std(cv_results_2y['rmse']):.4f}")

# 10-fold CV for 5-year model
print("\n=== 5-Year Recovery Model ===")
for fold, (train_idx, test_idx) in enumerate(kf.split(X_5y), 1):
    X_train, X_test = X_5y.iloc[train_idx], X_5y.iloc[test_idx]
    y_train, y_test = y_5y.iloc[train_idx], y_5y.iloc[test_idx]
    
    rf_5y.fit(X_train, y_train)
    y_pred = rf_5y.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    cv_results_5y["r2"].append(r2)
    cv_results_5y["rmse"].append(rmse)
    cv_results_5y["mae"].append(mae)
    
    print(f"Fold {fold}: R² = {r2:.3f}, RMSE = {rmse:.4f}, MAE = {mae:.4f}")

print(f"\nMean R² = {np.mean(cv_results_5y['r2']):.3f} ± {np.std(cv_results_5y['r2']):.3f}")
print(f"Mean RMSE = {np.mean(cv_results_5y['rmse']):.4f} ± {np.std(cv_results_5y['rmse']):.4f}")

# Fit final models on full data
rf_2y.fit(X_2y, y_2y)
rf_5y.fit(X_5y, y_5y)


=== 2-Year Recovery Model ===
Fold 1: R² = 0.376, RMSE = 0.0974, MAE = 0.0712
Fold 2: R² = 0.309, RMSE = 0.1076, MAE = 0.0728
Fold 3: R² = 0.312, RMSE = 0.1073, MAE = 0.0743
Fold 4: R² = 0.404, RMSE = 0.1006, MAE = 0.0724
Fold 5: R² = 0.341, RMSE = 0.1020, MAE = 0.0714
Fold 6: R² = 0.363, RMSE = 0.0951, MAE = 0.0703
Fold 7: R² = 0.371, RMSE = 0.0965, MAE = 0.0718
Fold 8: R² = 0.343, RMSE = 0.1052, MAE = 0.0720
Fold 9: R² = 0.378, RMSE = 0.0956, MAE = 0.0706
Fold 10: R² = 0.313, RMSE = 0.1167, MAE = 0.0773

Mean R² = 0.351 ± 0.031
Mean RMSE = 0.1024 ± 0.0065

=== 5-Year Recovery Model ===
Fold 1: R² = 0.469, RMSE = 0.0872, MAE = 0.0650
Fold 2: R² = 0.447, RMSE = 0.1010, MAE = 0.0756
Fold 3: R² = 0.429, RMSE = 0.0923, MAE = 0.0700
Fold 4: R² = 0.556, RMSE = 0.0865, MAE = 0.0664
Fold 5: R² = 0.489, RMSE = 0.0881, MAE = 0.0665
Fold 6: R² = 0.464, RMSE = 0.0966, MAE = 0.0728
Fold 7: R² = 0.456, RMSE = 0.0947, MAE = 0.0719
Fold 8: R² = 0.476, RMSE = 0.0941, MAE = 0.0717
Fold 9: R² = 0.419, 

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

Your Random Forest results show **moderate but acceptable performance** for ecological modeling:

## **2-Year Recovery Model:**
- **R² = 0.351** → Explains ~35% of variance
- **RMSE = 0.102** → Average error ±0.10 EVI units
- **MAE = 0.072** → Typical error ~0.07 EVI units
- **CV stability**: std = 0.031 (reasonable)

**Assessment**: **Fair to moderate**. The model captures some patterns but misses ~65% of variance.

## **5-Year Recovery Model:**
- **R² = 0.469** → Explains ~47% of variance
- **RMSE = 0.093** → Average error ±0.09 EVI units  
- **MAE = 0.069** → Typical error ~0.07 EVI units
- **CV stability**: std = 0.036 (good)

**Assessment**: **Moderate**. Better than 2-year model, captures nearly half the variance.

---

## **Interpretation:**

✅ **What's working:**
- 5-year model > 2-year model (longer recovery is more predictable)
- Low CV variance → stable, not overfitting
- RMSE/MAE reasonable for EVI difference scale

⚠️ **What's limiting:**
- R² < 0.5 suggests **high unexplained variance**
- Likely missing key factors:
  - **Weather/rainfall** (not included)
  - **Soil conditions** 
  - **Fire intensity details**
  - **Species composition**
  - **Local management**

---

## **Next steps to improve:**

````python
## Model accuracy
import matplotlib.pyplot as plt

# Plot CV results comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ["r2", "rmse", "mae"]
titles = ["R² Score", "RMSE", "MAE"]

for ax, metric, title in zip(axes, metrics, titles):
    ax.boxplot([cv_results_2y[metric], cv_results_5y[metric]], labels=["2-year", "5-year"])
    ax.set_title(title)
    ax.set_ylabel(title)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Check feature importance
print("\n=== Feature Importance ===")
feat_imp_2y = pd.DataFrame({
    "feature": predictor_cols,
    "importance_2y": rf_2y.feature_importances_
}).sort_values("importance_2y", ascending=False)

feat_imp_5y = pd.DataFrame({
    "feature": predictor_cols,
    "importance_5y": rf_5y.feature_importances_
}).sort_values("importance_5y", ascending=False)

print("\n2-Year Model:")
print(feat_imp_2y)
print("\n5-Year Model:")
print(feat_imp_5y)
````

**Your models are usable** but explain <50% variance—typical for complex ecological recovery where many unmeasured factors matter.

## Model accuracy

## Variable importance

## Partial dependence plots